### 0. Data Acquisition

Week 5 must compare the learned model against the Week-4 baseline on the **same data and same metric**. Therefore this notebook uses the same 30,000-row starter dataset used by Week 4: `data/raw/content_refresh_anonymized.csv`.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/pretom26/ml_internship_flyrankAI.git"
REPO_DIR = Path("/content/ml_internship_flyrankAI")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            check=True
        )
    os.chdir(REPO_DIR)
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            os.chdir(candidate)
            break

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Make sure the repository contains the Week-4 starter dataset."
    )

print(f"Using the Week-4 dataset: {DATA_PATH}")

# Load the same dataset used by Week 4.
df = pd.read_csv(DATA_PATH)

# Use the observed Week-4 label.
# trend_direction is allowed to define the evaluation label, but is NOT a feature.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(np.int8)
)

print(f"{len(df):,} rows")
print(f"{df['client_id'].nunique():,} clients")
print(f"Decline base rate: {df['is_declining_label'].mean():.3f}")

Using the Week-4 dataset: data/raw/content_refresh_anonymized.csv
30,000 rows
32 clients
Decline base rate: 0.542


# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Logistic Regression.**

This lane is a ranking problem: the goal is to prioritize which content a human should investigate first. Logistic Regression is a simple first learned model that produces a probability score, which can be ranked and compared directly with the Week-4 heuristic.

I chose it because it is interpretable, reproducible, and less complex than a tree ensemble. The model will use only decision-time fields. The observed Week-4 label `is_declining_label` is the target, while `trend_direction` and `trend_pct` are excluded from the feature set because they define the label.

The key comparison is not raw accuracy. The model and the Week-4 baseline will be evaluated on the **same held-out rows using Precision@K**, matching the Week-4 ranking metric.

In [2]:
import gc
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

# The starter dataset is small enough for pandas, but keep only fields needed here.
needed = [
    "client_id",
    "content_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_type",
    "position_tier",
]
available = [c for c in needed if c in df.columns]
df = df[available].copy()

# Week-4 gotcha: avg_position == 0 means no data, not rank zero.
df["avg_position_no_data"] = (
    df["avg_position"].fillna(0) == 0
).astype(np.int8)

df["avg_position_model"] = df["avg_position"].replace(0, np.nan)

# Compact / stable dtypes.
for col in ["days_since_last_update", "impressions_90d"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce", downcast="integer")

df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce")
df["avg_position_model"] = pd.to_numeric(
    df["avg_position_model"], errors="coerce"
)

# If position_tier was not already present, recreate the Week-4 tiers.
if "position_tier" not in df.columns:
    def get_tier(pos):
        if pd.isna(pos) or pos == 0:
            return "no_data"
        if pos <= 3:
            return "top_3"
        if pos <= 10:
            return "top_10"
        return "other"

    df["position_tier"] = df["avg_position"].map(get_tier)

# Decision-time model features.
# IDs are for grouping only; trend fields are label-derived and excluded.
FEATURES_NUMERIC = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position_model",
    "avg_position_no_data",
]

FEATURE_CATEGORICAL = ["content_type"]

FORBIDDEN = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
}

assert not (set(FEATURES_NUMERIC + FEATURE_CATEGORICAL) & FORBIDDEN)

print(f"Rows available for modeling: {len(df):,}")
print(f"Model features: {FEATURES_NUMERIC + FEATURE_CATEGORICAL}")
print(f"Model base rate: {df['is_declining_label'].mean():.3f}")

gc.collect()

Rows available for modeling: 30,000
Model features: ['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position_model', 'avg_position_no_data', 'content_type']
Model base rate: 0.542


0

## 2. Split design

I use a reproducible **80/20 grouped holdout by `client_id`**. This keeps all content from a client on one side of the split, so the model is evaluated on clients it did not train on.

The Week-4 baseline and Logistic Regression will be scored on the **exact same test rows**. The main metric is **Precision@K at 10, 20, 50, and 100**, the same ranking metric used by Week 4.

This split is a validation design, not a claim that the model has seen future time periods. The label is the observed `is_declining_label` supplied by the starter dataset.

In [3]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

train_clients = set(train["client_id"])
test_clients = set(test["client_id"])

assert train_clients.isdisjoint(test_clients)

print(f"Train rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"Train clients: {len(train_clients):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Overlapping clients: {len(train_clients & test_clients)}")
print(f"Train decline rate: {train['is_declining_label'].mean():.3f}")
print(f"Test decline rate: {test['is_declining_label'].mean():.3f}")

del train_idx, test_idx
gc.collect()

Train rows: 23,837
Test rows: 6,163
Train clients: 25
Test clients: 7
Overlapping clients: 0
Train decline rate: 0.550
Test decline rate: 0.511


0

## 3. Train + compare vs my baseline

The Week-4 baseline is reproduced as the same three-condition heuristic:

1. stale for 90+ days,
2. at least 500 impressions,
3. CTR below the median for its position tier.

The queue is ranked by impressions once all three conditions are satisfied.

The Logistic Regression model uses the same decision-time signals as its feature space plus content type. Both methods are evaluated on the **same test rows and the same Precision@K metrics**. No label-derived fields are used as features.

In [4]:
# ---------- Train + compare against the Week-4 baseline ----------

# 1. Reproduce the Week-4 rule on the same held-out test rows.
# The rule itself is unchanged from Week 4.
pos_median_ctr = (
    train.loc[train["position_tier"] != "no_data"]
    .groupby("position_tier")["ctr"]
    .median()
)

expected_ctr = test["position_tier"].map(pos_median_ctr)

stale = (test["days_since_last_update"] >= 90).astype(np.int8)
visible = (test["impressions_90d"] >= 500).astype(np.int8)

ctr_underperform = (
    (test["position_tier"] != "no_data") &
    (test["ctr"] < expected_ctr)
).astype(np.int8)

baseline_score = (
    stale *
    visible *
    ctr_underperform *
    test["impressions_90d"].fillna(0)
)

# 2. Logistic Regression.
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, FEATURES_NUMERIC),
    ("cat", categorical_pipe, FEATURE_CATEGORICAL),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(
        max_iter=1000,
        random_state=SEED
    )),
])

X_train = train[FEATURES_NUMERIC + FEATURE_CATEGORICAL]
X_test = test[FEATURES_NUMERIC + FEATURE_CATEGORICAL]
y_train = train["is_declining_label"]
y_test = test["is_declining_label"].to_numpy()

model.fit(X_train, y_train)

test["model_score"] = model.predict_proba(X_test)[:, 1]
test["baseline_score"] = baseline_score

# 3. Same ranking metric as Week 4.
def precision_at_k(scores, labels, k):
    k = min(k, len(labels))
    order = np.argsort(-np.asarray(scores), kind="mergesort")[:k]
    return np.asarray(labels)[order].mean()

ks = [10, 20, 50, 100]

results = []

for k in ks:
    results.append({
        "metric": f"Precision@{k}",
        "Week-4 baseline": precision_at_k(
            test["baseline_score"], y_test, k
        ),
        "Logistic Regression": precision_at_k(
            test["model_score"], y_test, k
        ),
    })

comparison = pd.DataFrame(results)

print("MODEL VS WEEK-4 BASELINE")
display(comparison.round(3))

print(f"Test decline base rate: {y_test.mean():.3f}")
print(
    f"Logistic Regression ROC-AUC (supplementary): "
    f"{roc_auc_score(y_test, test['model_score']):.3f}"
)

p20_baseline = precision_at_k(
    test["baseline_score"], y_test, 20
)
p20_model = precision_at_k(
    test["model_score"], y_test, 20
)

print(
    f"Precision@20 difference (model - baseline): "
    f"{p20_model - p20_baseline:+.3f}"
)

del X_train, X_test, y_train, baseline_score
gc.collect()

MODEL VS WEEK-4 BASELINE


,metric,Week-4 baseline,Logistic Regression
0,Precision@10,0.40,0.50
1,Precision@20,0.50,0.55
2,Precision@50,0.52,0.62
3,Precision@100,0.53,0.60


Test decline base rate: 0.511
Logistic Regression ROC-AUC (supplementary): 0.563
Precision@20 difference (model - baseline): +0.050


54

## 4. Errors and interpretation

I will read the result before treating the score as a win:

1. **Ranking performance:** does Logistic Regression beat the Week-4 rule at the top of the queue?
2. **Feature interpretation:** which signals receive the strongest coefficients, and do their directions make sense?
3. **Error review:** inspect concrete false positives and observed declines that the model ranks poorly.

A higher score is only useful if the errors are understandable and the model is using decision-time signals rather than a shortcut to the label.

In [5]:
# ---------- Errors and interpretation ----------

feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["logreg"].coef_[0]

coef_table = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
    })
    .assign(abs_coefficient=lambda x: x["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
)

print("Top Logistic Regression coefficients:")
display(coef_table.head(10).round(3))

review_cols = [
    "content_id",
    "client_id",
    "model_score",
    "is_declining_label",
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "position_tier",
]

ranked = test.sort_values(
    ["model_score", "impressions_90d"],
    ascending=[False, False]
)

false_positives = ranked.loc[
    ranked["is_declining_label"] == 0,
    review_cols
].head(3)

hard_declines = ranked.loc[
    ranked["is_declining_label"] == 1,
    review_cols
].tail(3)

print("\nThree concrete false positives:")
display(false_positives.round(3))

print("\nThree low-ranked observed declines:")
display(hard_declines.round(3))

print("\nInterpretation:")
for rank, row in coef_table.head(3).reset_index(drop=True).iterrows():
    direction = (
        "higher decline probability"
        if row["coefficient"] > 0
        else "lower decline probability"
    )
    print(
        f"{rank + 1}. {row['feature']}: "
        f"{row['coefficient']:+.3f} ({direction})"
    )

print("""
The main decision-support metric is Precision@K because this is a ranked review queue.

The model is evaluated on exactly the same held-out rows and the same Precision@K
metrics as the Week-4 heuristic baseline.

The observed label comes from the starter dataset; trend_direction and trend_pct
are not model features because they are label-derived.

False positives show cases where the available decision-time signals look risky
but the observed decline label is 0. Low-ranked declines show cases the available
signals did not prioritize.

These coefficients are directional associations, not causal effects.
""")

del ranked, false_positives, hard_declines, feature_names, coefs
gc.collect()

Top Logistic Regression coefficients:


,feature,coefficient,abs_coefficient
4,num__avg_position_no_data,-1.105,1.105
6,cat__content_type_feedly article,-0.353,0.353
2,num__ctr,-0.240,0.240
3,num__avg_position_model,-0.212,0.212
7,cat__content_type_keyword article,0.190,0.190
0,num__days_since_last_update,0.137,0.137
1,num__impressions_90d,-0.124,0.124
5,cat__content_type_comparison article,0.075,0.075



Three concrete false positives:


,content_id,client_id,model_score,is_declining_label,days_since_last_update,impressions_90d,ctr,avg_position,position_tier
19205,content_17c2de35424d,client_8527a891e2,0.696,0,103,3,0.0,1.0,top_3
7256,content_bc4411bf8d9f,client_8527a891e2,0.695,0,104,24,0.0,1.6,top_3
27709,content_84c5f6af423c,client_8527a891e2,0.694,0,104,8,0.0,1.9,top_3



Three low-ranked observed declines:


,content_id,client_id,model_score,is_declining_label,days_since_last_update,impressions_90d,ctr,avg_position,position_tier
6653,content_5fe46e04994d,client_4e07408562,0.044,1,104,517715,0.14,4.2,page_1
26844,content_8c19996aa890,client_4e07408562,0.037,1,20,509252,0.15,2.5,top_3
27271,content_7bc32bc1df59,client_8527a891e2,0.011,1,92,1,0.00,0.0,top_3



Interpretation:
1. num__avg_position_no_data: -1.105 (lower decline probability)
2. cat__content_type_feedly article: -0.353 (lower decline probability)
3. num__ctr: -0.240 (lower decline probability)

The main decision-support metric is Precision@K because this is a ranked review queue.

The model is evaluated on exactly the same held-out rows and the same Precision@K
metrics as the Week-4 heuristic baseline.

The observed label comes from the starter dataset; trend_direction and trend_pct
are not model features because they are label-derived.

False positives show cases where the available decision-time signals look risky
but the observed decline label is 0. Low-ranked declines show cases the available
signals did not prioritize.

These coefficients are directional associations, not causal effects.



93

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.